In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_path = "/kaggle/input/kermany2018/OCT2017 "
train_dir, val_dir, test_dir = os.path.join(dataset_path, 'train'), os.path.join(dataset_path, 'val'), os.path.join(dataset_path, 'test')

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet50 expects 224x224 images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_full = datasets.ImageFolder(root=train_dir, transform=image_transform)
val_dataset_from_val_folder = datasets.ImageFolder(root=val_dir, transform=image_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=image_transform)

# Adjust train-test split (70-15-15)
train_indices, temp_indices = train_test_split(
    np.arange(len(train_dataset_full)),
    test_size=0.3,
    stratify=train_dataset_full.targets,
    random_state=42
)

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.5,
    stratify=[train_dataset_full.targets[i] for i in temp_indices],
    random_state=42
)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(train_dataset_full, val_indices)
test_dataset = Subset(train_dataset_full, test_indices)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, prefetch_factor=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, prefetch_factor=2)

print(f"Train size: {len(train_loader.dataset)}, Val size: {len(val_loader.dataset)}, Test size: {len(test_loader.dataset)}")


class ResNet50Classifier(nn.Module):
    """ResNet50 Model for Retinal OCT Classification"""
    def __init__(self, num_classes):
        super().__init__()
        self.resnet = timm.create_model("resnet50", pretrained=True, num_classes=num_classes)

    def forward(self, x):
        return self.resnet(x)


def save_model(model, optimizer, filename="resnet50_classifier.pth"):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, filename)
    print(f"Model saved to {filename}")

def load_model(filename="resnet50_classifier.pth"):
    model = ResNet50Classifier(num_classes=len(train_dataset_full.classes)).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    
    checkpoint = torch.load(filename, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    print(f"Model loaded from {filename}")
    return model, optimizer


num_classes = len(train_dataset_full.classes)
model = ResNet50Classifier(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []

def train_model(model, train_loader, val_loader, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        train_loss, correct_train, total_train = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            correct_train += predicted.eq(labels).sum().item()
            total_train += labels.size(0)

        train_acc = correct_train / total_train
        train_losses.append(train_loss / len(train_loader))
        train_accuracies.append(train_acc)

        # Validation
        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                correct_val += predicted.eq(labels).sum().item()
                total_val += labels.size(0)

        val_acc = correct_val / total_val
        val_losses.append(val_loss / len(val_loader))
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_losses[-1]:.4f}, Train Acc: {train_accuracies[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Val Acc: {val_accuracies[-1]:.4f}")

train_model(model, train_loader, val_loader, num_epochs=10)
save_model(model, optimizer)


class_names = train_dataset_full.classes

def evaluate_model(model, loader):
    """Evaluate and Generate Classification Report"""
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_percent, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.title("Confusion Matrix")
    plt.show()

evaluate_model(model, test_loader)

Using device: cuda


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Train size: 58438, Val size: 12523, Test size: 12523


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Epoch [1/10], Train Loss: 0.2674, Train Acc: 0.9053, Val Loss: 0.1318, Val Acc: 0.9529
Epoch [2/10], Train Loss: 0.1349, Train Acc: 0.9538, Val Loss: 0.1352, Val Acc: 0.9534


KeyboardInterrupt: 